In [1]:
import os
import certifi
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub


c:\Users\dell\anaconda3\envs\langagent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain.agents import create_react_agent,AgentExecutor

In [3]:
#=======================================
#LOAD ENV VARIABLES
#=======================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPEN_API_KEY = os.getenv("OPEN_API_KEY") or os.getenv("OPENAI_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [4]:
# Create the search tool only when Tavily is configured.
search_tool = None
if TAVILY_API_KEY:
    search_tool = TavilySearchResults(
        max_results=3,
        tavily_api_key=TAVILY_API_KEY,
    )
else:
    print("TAVILY_API_KEY is missing. Responses will work without web references.")

In [5]:
# Optional live search smoke test. Disabled when Tavily is not configured.
if search_tool is None:
    print("Search smoke test skipped. Add TAVILY_API_KEY to .env to enable it.")
else:
    print(search_tool.invoke("What is the latest news about AI?"))

[{'url': 'https://www.reddit.com/r/AISEOInsider/comments/1rs1o1v/latest_ai_news_the_biggest_ai_announcements_right', 'content': 'This allows smaller companies to build marketing campaigns without large marketing teams.\n\nTools like Pomelli show how AI marketing platforms are evolving quickly.\n\n# Anthropic Lawsuit Dominates Latest AI News\n\nOne of the biggest stories in the latest AI news involves Anthropic.\n\nAnthropic develops the Claude AI model family.\n\nThe company filed lawsuits against the US government after being placed on a technology blacklist.\n\nThe disagreement began over how AI models could be used in defense systems.\n\nAnthropic refused to allow its models to support mass surveillance or autonomous weapons.\n\nNegotiations between the company and government agencies eventually broke down.\n\nFederal agencies were instructed to stop using Anthropic technology.\n\nAnthropic responded by filing legal challenges. [...] Anthropic responded by filing legal challenges.\n

In [6]:
%pip install -q "langchain-google-genai==0.0.11" "langchain-core==0.1.42"

Note: you may need to restart the kernel to use updated packages.


In [7]:
#===============================
# Gemini 3.5 Flash-Lite model
#===============================
llm = None
if GOOGLE_API_KEY:
    llm = ChatGoogleGenerativeAI(
        model="gemini-3.5-flash-lite",
        google_api_key=GOOGLE_API_KEY,
        temperature=0,
        max_retries=0,
    )
else:
    print("Set GOOGLE_API_KEY in .env to use the Gemini model.")


In [8]:
#===============================
# Check model configuration without spending API quota.
#===============================
if llm is None:
    print("Gemini: not configured (GOOGLE_API_KEY missing)")
else:
    print("Gemini 3.5 Flash-Lite: configured and ready")


Gemini 3.5 Flash-Lite: configured and ready


In [9]:
# Optional smoke test. Disabled by default to avoid consuming API quota.
RUN_LIVE_TESTS = os.getenv("RUN_LIVE_TESTS", "0") == "1"
if not RUN_LIVE_TESTS:
    response = {"output": "Smoke test skipped. Set RUN_LIVE_TESTS=1 to call Gemini."}
elif llm is None:
    response = {"output": "Gemini is not configured. Add GOOGLE_API_KEY to .env."}
else:
    try:
        response = llm.invoke("Reply with exactly: Gemini is working")
        print(getattr(response, "content", response))
    except Exception as error:
        print(f"Gemini smoke test failed: {error}")
        response = {"output": "Gemini smoke test failed. Check the API key, model access, or quota."}


In [10]:
# Load the prompt required by the ReAct agent.
react_prompt = hub.pull("hwchase17/react")

c:\Users\dell\anaconda3\envs\langagent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [11]:
# Tools used by the agent. An empty list is valid when web search is unavailable.
tools = [search_tool] if search_tool is not None else []

In [12]:
#======================================
# CREATE AGENT
#======================================
agent = None
if llm is not None:
    agent = create_react_agent(
        llm=llm,
        tools=tools,
        prompt=react_prompt
    )
else:
    print("Agent not created because no LLM is configured.")

In [13]:
#=====================
# EXECUTOR
#=====================
agent_executor = None
if agent is not None:
    agent_executor = AgentExecutor(
        agent=agent,
        tools=tools,
        verbose=True,
        handle_parsing_errors=True,
    )
else:
    print("Executor not created because no agent is available.")


In [14]:
#====================
# RUN
#====================
if not RUN_LIVE_TESTS:
    response = {"output": "Demo run skipped. Set RUN_LIVE_TESTS=1 to call Gemini."}
elif agent_executor is None:
    response = {"output": "No LLM configured. Add GOOGLE_API_KEY and rerun the setup cells."}
else:
    try:
        response = agent_executor.invoke({
            "input": "Tell me the news about pm modi"
        })
    except Exception as error:
        print(f"Agent execution failed: {error}")
        response = {"output": "Agent execution failed. Check the API key, model access, or quota."}


In [15]:
if isinstance(response, dict):
    print(response.get("output", response))
else:
    print(getattr(response, "content", response))

Demo run skipped. Set RUN_LIVE_TESTS=1 to call Gemini.


In [16]:
def get_response(prompt, max_results=10, custom_instructions=''):
    """Return the answer and reference links for a user prompt."""
    return answer_question(
        prompt,
        max_results=max_results,
        custom_instructions=custom_instructions,
    )


In [ ]:
#===============================
# Small question-and-references interface
#===============================
import gradio as gr


def answer_question(question, max_results=10, custom_instructions=''):
    question = (question or "").strip()
    if not question:
        return "Please enter a question.", ""
    if llm is None:
        return "Gemini is not configured. Add GOOGLE_API_KEY to .env and rerun the setup cells.", ""
    if search_tool is None:
        return "Web search is not configured. Add TAVILY_API_KEY to .env and rerun the setup cells.", ""

    try:
        max_results = max(1, min(int(max_results), 10))
        search_results = search_tool.invoke(question)
        if not isinstance(search_results, list):
            search_results = []

        sources = []
        context_parts = []
        for result in search_results[:max_results]:
            url = result.get("url", "")
            content = result.get("content", "")
            if url:
                sources.append(url)
                context_parts.append(f"Source: {url}\n{content}")

        context = "\n\n".join(context_parts) or "No web sources were found."
        instructions = custom_instructions.strip()
        prompt = (
            "Answer the user's question using the web sources below. "
            "Be concise, factual, and say when the sources do not provide enough evidence.\n\n"
            f"{instructions}\n\n" if instructions else ""
        ) + f"Question: {question}\n\nWeb sources:\n{context}"
        model_response = llm.invoke(prompt)
        answer = getattr(model_response, "content", str(model_response))
        references = "\n".join(
            f"{index}. [{url}]({url})" for index, url in enumerate(dict.fromkeys(sources), 1)
        )
        return answer, references or "No reference links were returned."
    except Exception as error:
        return f"Could not answer the question: {error}", ""


with gr.Blocks(title="Ask with References") as gradio_app:
    gr.Markdown("# Ask with References\nAsk a question and get one answer supported by web sources.")
    question_box = gr.Textbox(
        label="Question",
        placeholder="Ask a question about current events, technology, or any topic...",
        lines=3,
    )
    max_results_box = gr.Slider(
        minimum=1,
        maximum=10,
        value=3,
        step=1,
        label="Number of links",
    )
    ask_button = gr.Button("Ask", variant="primary")
    answer_box = gr.Textbox(label="Answer", lines=10)
    references_box = gr.Markdown(label="References")

    ask_button.click(
        fn=answer_question,
        inputs=[question_box, max_results_box],
        outputs=[answer_box, references_box],
    )
    question_box.submit(
        fn=answer_question,
        inputs=[question_box, max_results_box],
        outputs=[answer_box, references_box],
    )

gradio_app.launch(share=False, prevent_thread_lock=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## How this agent works in the real world

This notebook demonstrates a practical question-answering agent that can be connected to a web application or an internal service.

1. **Configuration:** API keys are loaded from environment variables or a local `.env` file. In production, store them in a secret manager and never commit the `.env` file.
2. **User request:** A user sends a question to `answer_question()` through a UI, API endpoint, or background job.
3. **Tool use:** When Tavily is configured, the agent searches the web for current information and collects reference links. Without a search key, the notebook still runs in demo mode.
4. **Reasoning and response:** Gemini interprets the question, decides whether a search is needed, and writes a final answer using the available context.
5. **Application integration:** A FastAPI or Streamlit frontend can call the same function and return the answer to a user. In a production service, add authentication, input validation, request timeouts, logging, retries, and rate limits.
6. **Monitoring:** Track latency, token usage, search failures, and user feedback. Keep live tests disabled by default during development to avoid unexpected API costs.

A typical production flow is:

`user question -> application endpoint -> agent -> search tool (optional) -> Gemini -> answer and sources`

The notebook is intentionally safe to run without credentials: it reports configuration status and uses a local demo response when live tests are disabled.